# Feature group: `school`

Which students are *excellent*, and how do the school-related features differ between excellent and
non-excellent students? The notebook first looks at the data, then trains a decision tree and a random forest on
the features of this group and counts their mistakes on students they have never seen.

Open it with `uv run jupyter lab` from the repository root, or pick the project `.venv` kernel.
If `data/raw/` is empty, run `uv run python scripts/download_data.py` first (notebook 01 does this too).

In [ ]:
import matplotlib.pyplot as plt

from data_use_ce.data import TARGET, add_excellence, load_students
from data_use_ce.features import load_feature_groups
from data_use_ce.plots import (
    plot_excellence_rate,
    plot_feature_grid,
    plot_feature_histogram,
    plot_scatter,
)

DATASET = "mat"  # "mat" = maths course, "por" = Portuguese course
EXCELLENCE_THRESHOLD = 14  # a student is excellent if the final grade G3 is >= this value
GROUP = "school"  # a key of configs/feature_groups.yaml

df = add_excellence(load_students(DATASET), EXCELLENCE_THRESHOLD)
features = load_feature_groups(df.columns)[GROUP]

n_excellent = int(df[TARGET].sum())
print(
    f"{len(df)} students (final grade 0 excluded); "
    f"{n_excellent} excellent ({n_excellent / len(df):.0%}) with G3 >= {EXCELLENCE_THRESHOLD}"
)
print(f"Group '{GROUP}': {features}")

## The features

| Column | Meaning |
|---|---|
| `failures` | number of past class failures (0 to 3; 3 means 3 or more) |
| `absences` | number of school absences |
| `studytime` | weekly study time: 1 = under 2 h, 2 = 2 to 5 h, 3 = 5 to 10 h, 4 = over 10 h |
| `schoolsup` | extra educational support from the school (yes/no) |
| `paid` | extra paid classes in the subject (yes/no) |
| `activities` | extra-curricular activities (yes/no) |
| `higher` | wants to take higher education (yes/no) |
| `reason` | reason for choosing this school: close to `home`, `reputation`, `course` preference, `other` |
| `traveltime` | home to school travel time: 1 = under 15 min, 2 = 15 to 30 min, 3 = 30 to 60 min, 4 = over 1 h |

## Excellent vs not excellent, feature by feature

How to read the plots:

- **Bars** (one per value of the feature): the bar is split into the share of excellent students
  (orange) and the rest (blue). The label gives that share and `n`, the number of students with that value.
- **Dashed line**: the share of excellent students among *all* students. Orange above the line means
  that group does better than average.
- **Histogram** (`absences`, which has many values): each bar is the share *of its own class*
  that falls in the interval, orange on the left and blue on the right, so the small excellent class
  and the large other class can be compared.
- Be careful with small `n`: 0% out of 7 students says very little.

In [ ]:
fig = plot_feature_grid(df, features)
plt.show()

## A closer look

One feature at a time, and two features together (with a small random shift, `jitter`,
so that students with the same whole-number values do not hide each other).

In [ ]:
plot_excellence_rate(df, "higher")
plt.show()

In [ ]:
plot_feature_histogram(df, "absences", bins=25)
plt.show()

In [ ]:
plot_scatter(df, "studytime", "failures", jitter=0.15)
plt.show()

## Training models

The students are split in two: the model learns from the *training* students, and is judged on the *test* students, which it has never seen. The model gives each test student a score; the students with the highest scores are accepted. To keep the comparison simple, the model accepts as many students as there are excellent ones (`admit_fraction`).

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeClassifier

from data_use_ce.evaluation import (
    admit_top_fraction,
    count_mistakes,
    decision_table,
    random_selection_table,
)
from data_use_ce.modeling import make_preprocessor, split_students

SEED = 0
TEST_SIZE = 0.3  # share of students kept aside to test the model

X_train, X_test, y_train, y_test = split_students(df, features, TEST_SIZE, SEED)
admit_fraction = y_train.mean()  # accept as many students as there are excellent ones

print(
    f"Training: {len(X_train)} students ({y_train.sum()} excellent). "
    f"Test: {len(X_test)} students ({y_test.sum()} excellent)."
)

In [ ]:
models = {
    "Decision tree (depth 3)": DecisionTreeClassifier(max_depth=3, random_state=SEED),
    "Random forest": RandomForestClassifier(
        n_estimators=200, min_samples_leaf=3, random_state=SEED, n_jobs=-1
    ),
}

for name, estimator in models.items():
    model = make_pipeline(make_preprocessor(df, features), estimator)
    model.fit(X_train, y_train)  # the model only sees the training students
    scores = model.predict_proba(X_test)[:, 1]
    admitted = admit_top_fraction(scores, admit_fraction, seed=SEED)
    mistakes = count_mistakes(y_test, admitted)

    print(f"{name}: accepted {admitted.sum()} of {len(admitted)} test students")
    display(decision_table(y_test, admitted))
    print(
        f"  good students rejected: {mistakes['good_rejected']} of {mistakes['n_excellent']} "
        f"({mistakes['good_rejected_rate']:.0%})"
    )
    print(
        f"  bad students accepted:  {mistakes['bad_accepted']} of {mistakes['n_not_excellent']} "
        f"({mistakes['bad_accepted_rate']:.0%})\n"
    )

print("Picking the same number of students at random would give, on average:")
display(random_selection_table(y_test, int(admitted.sum())).round(1))

### Reading the tables

Rows are what the student really did (excellent or not), columns are what the model decided. A **good student rejected** is in the *Excellent / Rejected* cell; a **bad student accepted** is in the *Not excellent / Accepted* cell.

Since the model accepts about as many students as there are excellent ones, the two mistakes are about equal in number: every good student rejected leaves a place to a bad one. The random lottery at the end is the yardstick: a useful model rejects fewer good students than chance does.

This is a single random split with few excellent students in the test set, so change `SEED` and see how much the counts move.

## Your turn

Change `DATASET`, `EXCELLENCE_THRESHOLD` or `GROUP` in the first cell and run everything again.

In [ ]:
# Free exploration